In [ ]:
# Packages
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import geopandas as gpd
from pathlib import Path
import getpass

user = getpass.getuser()
path_users = Path.home()
path_tiger = Path(r'I:\Projects\Josh\Geospatial Data\TIGER')
path_cap = Path(r'I:\Projects\Josh\Regional Monitoring\ArcPro_sup\Cap to Cap\_data')

In [ ]:
years = [2020]

for year in tqdm(years):
    file_in = path_tiger / 'raw' / f'tl_{year}_us_zcta520' / f'tl_{year}_us_zcta520.shp'
    gdf_zip = gpd.read_file(file_in)
    gdf_zip = gdf_zip.to_crs("EPSG:2226")
    gdf_zip['ZCTA5CE20'] = gdf_zip['ZCTA5CE20'].astype(int)
    CA_zips = range(90001, 96162)
    VA_zips = range(22001, 24658)
    gdf_zip = gdf_zip[(gdf_zip['ZCTA5CE20'].isin(CA_zips)) | (gdf_zip['ZCTA5CE20'].isin(VA_zips))]
    gdf_zip = gdf_zip[['ZCTA5CE20', 'geometry']]
    gdf_zip = gdf_zip.reset_index(drop=True)

    file_in = path_tiger / 'raw' / f'tl_{year}_us_county' / f'tl_{year}_us_county.shp'
    gdf_counties = gpd.read_file(file_in)
    gdf_counties = gdf_counties.to_crs("EPSG:2226")
    gdf_counties_CA = gdf_counties[(gdf_counties['STATEFP'].isin(['06'])) & (gdf_counties['COUNTYFP'].isin(['017', '061', '067', '101', '113', '115']))]
    gdf_counties_VA = gdf_counties[(gdf_counties['STATEFP'].isin(['51'])) & (gdf_counties['COUNTYFP'].isin(['013']))]
    gdf_counties = pd.concat([gdf_counties_CA, gdf_counties_VA])
    gdf_counties = gdf_counties[['STATEFP', 'COUNTYFP', 'NAME', 'geometry']]
    gdf_counties = gdf_counties.reset_index(drop=True)

    gdf_int = gpd.overlay(gdf_zip, gdf_counties, how='intersection')
    gdf_int = gdf_int[['STATEFP', 'COUNTYFP', 'NAME', 'ZCTA5CE20']]
    gdf_int.reset_index(drop=True)

    gdf_zip = gdf_zip.merge(gdf_int, on=['ZCTA5CE20'])
    gdf_zip = gdf_zip.sort_values(['STATEFP', 'COUNTYFP', 'ZCTA5CE20'])

    display(gdf_zip.head())

    file_json = path_cap / f'tl_{year}_zip_cap.geojson'
    file_shp = path_cap / f'tl_{year}_zip_cap' / f'tl_{year}_zip_cap.shp'
    gdf_zip.to_file(file_json, driver='GeoJSON')
    gdf_zip.to_file(file_shp)
    


